# 34 · Mapping reaction scales between platforms (E87)

Leave one platform out: train on two platforms, test on the third. Mappings use only the target's inputs, never its views, so they would apply to Facebook / Instagram.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 300)
import extra_models as em, history as H, platform_map as pm

In [2]:
f, meta = em.load(ROOT); P = pm.lopo(f, meta); S = H.summarise(P); display(S.pivot(index="model", columns="platform", values="MAE")[pm.PLATS].round(3)); display(S.pivot(index="model", columns="platform", values="bias")[pm.PLATS].round(3))

platform,tiktok,youtube,twitter
model,,,
"CatBoost, none",0.410,0.628,0.707
"CatBoost, quantile",0.822,0.651,1.358
"CatBoost, ratios",0.604,0.719,1.207
"CatBoost, z-score",1.044,0.654,1.429
"ridge, none",0.312,0.604,0.641
"ridge, quantile",0.808,0.613,1.286
"ridge, ratios",0.370,0.733,0.733
"ridge, z-score",0.756,0.649,1.284


platform,tiktok,youtube,twitter
model,,,
"CatBoost, none",-0.176,-0.130,0.523
"CatBoost, quantile",-0.757,-0.357,1.339
"CatBoost, ratios",-0.532,-0.169,1.177
"CatBoost, z-score",-1.023,0.173,1.418
"ridge, none",-0.053,-0.206,0.383
"ridge, quantile",-0.779,-0.273,1.255
"ridge, ratios",-0.083,-0.274,0.405
"ridge, z-score",-0.723,-0.156,1.255


In [3]:
G = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain over ridge, none", "CI low", "CI high"], H.gain_ci(P, p, "ridge, none", k)))} for p in pm.PLATS for k in P.model.unique() if k != "ridge, none"]); G.round(3)

,platform,model,"gain over ridge, none",CI low,CI high
0,tiktok,"CatBoost, none",-0.098,-0.112,-0.085
1,tiktok,"ridge, z-score",-0.444,-0.463,-0.425
2,tiktok,"CatBoost, z-score",-0.732,-0.760,-0.706
3,tiktok,"ridge, quantile",-0.496,-0.514,-0.475
4,tiktok,"CatBoost, quantile",-0.509,-0.534,-0.486
5,tiktok,"ridge, ratios",-0.058,-0.072,-0.044
6,tiktok,"CatBoost, ratios",-0.292,-0.312,-0.273
7,youtube,"CatBoost, none",-0.023,-0.038,-0.008
8,youtube,"ridge, z-score",-0.045,-0.063,-0.029
9,youtube,"CatBoost, z-score",-0.050,-0.080,-0.021


In [4]:
from scipy.stats import spearmanr
R = P.groupby(["platform", "model"]).apply(lambda q: spearmanr(q.y, q.pred)[0]).unstack(0)[pm.PLATS]; R.round(3)

platform,tiktok,youtube,twitter
model,,,
"CatBoost, none",0.862,0.679,0.541
"CatBoost, quantile",0.743,0.711,0.538
"CatBoost, ratios",0.847,0.616,0.593
"CatBoost, z-score",0.775,0.730,0.563
"ridge, none",0.861,0.654,0.495
"ridge, quantile",0.818,0.653,0.491
"ridge, ratios",0.852,0.624,0.376
"ridge, z-score",0.800,0.640,0.495


With a shift fitted on n target posts with views (E15).

In [5]:
W = pm.with_shift(P); W.pivot_table(index="model", columns=["platform", "n"], values="MAE").round(3)

platform           tiktok                      twitter                      youtube                     
n                      0      10     25     50      0      10     25     50      0      10     25     50
model                                                                                                   
CatBoost, none      0.410  0.433  0.415  0.409   0.707  0.590  0.573  0.566   0.628  0.666  0.647  0.635
CatBoost, quantile  0.822  0.481  0.466  0.460   1.358  0.581  0.560  0.548   0.651  0.644  0.623  0.616
CatBoost, ratios    0.604  0.438  0.427  0.419   1.207  0.546  0.528  0.521   0.719  0.756  0.733  0.723
CatBoost, z-score   1.044  0.504  0.482  0.476   1.429  0.550  0.532  0.524   0.654  0.665  0.646  0.639
ridge, none         0.312  0.331  0.320  0.316   0.641  0.582  0.560  0.552   0.604  0.638  0.619  0.610
ridge, quantile     0.808  0.357  0.345  0.340   1.286  0.589  0.565  0.557   0.613  0.638  0.615  0.608
ridge, ratios       0.370  0.392  0.380  0.375   0.733  0.685  0.657  0.644   0.733  0.773  0.749  0.735
ridge, z-score      0.756  0.381  0.368  0.363   1.284  0.595  0.570  0.560   0.649  0.688  0.661  0.654

In [6]:
json.dump({"models": S.to_dict("records"), "gains": G.to_dict("records"), "spearman": R.reset_index().to_dict("records"), "with_shift": W.to_dict("records")},
          open(ROOT / "data" / "processed" / "platform_map_numbers.json", "w"), indent=1, default=float)